In [67]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [68]:
def count_words_in_go_files(folder_path, words_dict):
    word_search_counts = {word: 0 for word in words_dict.keys()}
    word_markup_counts = words_dict.copy()
    
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, words_dict.keys())) + r')\b')
    
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.go'):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        matches = pattern.findall(content)
                        for word in matches:
                            word_search_counts[word] += 1
                except (UnicodeDecodeError, PermissionError) as e:
                    print(f"Ошибка при обработке файла {file_path}: {e}")
    
    # Создаем DataFrame
    df = pd.DataFrame({
        'Word': list(word_search_counts.keys()),
        'Count Search': list(word_search_counts.values()),
        'Count Markup': [word_markup_counts[word] for word in word_search_counts.keys()]
    })
    
    df['Count Search 2'] = np.ceil(df['Count Search'] / 2).astype(int)
    
    # Вычисляем Ratio
    df['Ratio'] = (df['Count Search 2']) / df['Count Markup']
    
    median = df['Ratio'].median()
    mad = (df['Ratio'] - median).abs().median()
    threshold = 3
    #df['IsOutlier_MAD'] = (df['Ratio'] - median).abs() > (threshold * mad)
    df['Count Search/2<20'] = df['Count Search 2']<20

    
    # Сортируем по Ratio (по убыванию)
    df_sorted = df.sort_values(by='Ratio', ascending=False)
    df_sorted.reset_index(drop=True, inplace=True)
    
    return df_sorted

In [69]:
global_array = np.array([])
one_array = np.array([])

In [70]:
def analyze(folder, words):
    result_df = count_words_in_go_files(folder, words)

    mean_row = pd.DataFrame({
        'Word': ['MEAN'],
        'Count Search 2': [result_df['Count Search 2'].mean()],
        'Count Markup': [result_df['Count Markup'].mean()],
        'Ratio': [result_df['Ratio'].mean()]
    })
    
    global global_array
    global_array = np.append(global_array, result_df['Ratio'].values)

    display(result_df)
    display(mean_row)
    iterate_over_column(result_df[result_df['Count Search/2<20']]["Count Search 2"], result_df[result_df['Count Search/2<20']]["Count Markup"])


In [137]:
# время выбора (просмотра варианта)
dt1 = 3000
# время выбора (просмотр варианта), когда есть ранжирование
dt2 = 2000
# время осознания 
dt = 500
# время на навигацию
dd = 100

In [138]:
def iterate_over_column(column1: pd.Series, column2: pd.Series):
    s1 = 0
    for val in column1:
        if val == 1:
            s1 += val * 0 + dt + dd
        else:
            s1 += val * dt1 + dt + dd
    
    s2 = 0
    for val in column2:
        if val == 1:
            s2 += val * 0 + dt + dd
        else:
            s2 += val * dt2 + dt + dd
    
    mu0 = s1/s2
    print(mu0)
    
    A = sum(column1)
    B = sum(column2)
    C = len(column1) * (dt + dd)
    
    mx = mu0
    mn = mu0
    k1 = A / (B * dt2 + C)
    k2 = (A * dt1 + C) * B / (B * dt2 + C)**2
    for delta1 in range(-300, 300, 10):
        for delta2 in range(-200, 200, 10):
            mu_delta = mu0 + k1 * delta1 - k2 * delta2
            mx = max(mx, mu_delta)
            mn = min(mn, mu_delta)
    print(k1, k2, mx, mn)

In [139]:
folder = 'e:\\phd\\test_repos_graphql2\\boost\\'
words = {
    "DealUpdate": 1, 
    "StorageAskUpdate": 1, 
    "FundsMoveToEscrow": 1, 
    "DealFailPaused": 1, 
    "DealRetryPaused": 1, 
    "DealCancel": 1, 
    "Mpool": 1, 
    "FundsLogs": 1, 
    "ProposalLogs": 1, 
    "LegacyDeals": 1, 
    //
    "Deals": 1, 
    "LegacyDeal": 1, 
    "Deal": 2, 
    "DealNew": 1, 
    "DealsCount": 1, 
    "StorageAsk": 1, 
    "DealPublishNow": 1, 
    "DealPublish": 1,
    "Epoch": 1,
    "Funds": 1,
    "ProposalLogsCount": 1,
    "SealingPipeline": 1,
    "Storage": 1,
    "Transfers": 1,
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Deals,56,1,28,28.0,False
1,StorageAsk,32,1,16,16.0,True
2,Storage,26,1,13,13.0,True
3,Deal,16,2,8,4.0,True
4,Epoch,7,1,4,4.0,True
5,ProposalLogs,5,1,3,3.0,True
6,FundsLogs,4,1,2,2.0,True
7,Transfers,4,1,2,2.0,True
8,Mpool,1,1,1,1.0,True
9,LegacyDeals,1,1,1,1.0,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,3.833333,1.041667,3.666667


8.865168539325843
0.001035598705501618 0.001293241587331511 9.424140481387616 8.30877302608237


In [126]:
folder = 'e:\\phd\\test_repos_graphql2\\circuit\\'
words = {
    "Secrets": 2, 
    "Services": 2, 
    "Releases": 2, 
    "EnvsDeployedIn": 2, 
    "Features": 2, 
    "GetGitCommits": 1, 
    "BookmarkProject": 1, 
    "UpdateProjectEnvironments": 1, 
    "UpdateUserPermissions": 1, 
    "DeleteProjectExtension": 1, 
    //
    "UpdateProjectExtension": 1, 
    "CreateProjectExtension": 1, 
    "DeleteExtension": 1, 
    "UpdateExtension": 1, 
    "CreateExtension": 1, 
    "ImportSecrets": 1, 
    "DeleteSecret": 1, 
    "UpdateSecret": 1, 
    "CreateSecret": 1, 
    "DeleteEnvironment": 1,
    "UpdateEnvironment": 1, 
    "CreateEnvironment": 1, 
    "DeleteServiceSpec": 1, 
    "UpdateServiceSpec": 1, 
    "CreateServiceSpec": 1, 
    "ImportServices": 1, 
    "DeleteService": 1, 
    "UpdateService": 1, 
    "CreateService": 1, 
    "StopRelease": 1,
    "CreateRelease": 1, 
    "UpdateProject": 1, 
    "CreateProject": 1, 
    "Permissions": 1, 
    "ReleaseExtensions": 1, 
    "ProjectExtensions": 1, 
    "Extensions": 1, 
    "Secret": 1, 
    "ExportSecrets": 1, 
    "Environments": 1, 
    "ServiceSpecs": 1, 
    "Users": 1, 
    "User": 1, 
    "Bookmarked": 1, 
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,User,2577,1,1289,1289.0,False
1,Secret,660,1,330,330.0,False
2,Extensions,379,1,190,190.0,False
3,Permissions,371,1,186,186.0,False
4,Services,541,2,271,135.5,False
5,CreateProject,180,1,90,90.0,False
6,CreateEnvironment,164,1,82,82.0,False
7,Secrets,299,2,150,75.0,False
8,CreateService,143,1,72,72.0,False
9,Users,122,1,61,61.0,False


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,72.659091,1.113636,67.363636


45.375
0.0025150602409638556 0.005993794454928147 47.30312636086515 43.481660981274494


In [127]:
folder = 'e:\\phd\\test_repos_graphql2\\core-geth\\'
words = {
    "SendRawTransaction": 2,
    "Logs": 2,
    "Transaction": 2,
    "Blocks": 1,
    "Block": 1,
    // 
    "EstimateGas": 1,
    "Call": 1,
    "Account": 1,
    "EstimateGas": 1,
    "Call": 1,
    "Account": 1,
    "OmmerAt": 1,
    "Miner": 1,
    "CreatedContract": 1,
    "Account": 1,
    "Storage": 1,
    "Status": 1,
    "Nonce": 1,
    "Hash": 1,
    "TransactionCount": 1,
    "ChainID": 1,
    "MaxPriorityFeePerGas": 1,
    "GasPrice": 1,
    "Transactions": 1,
    "ExcessBlobGas": 1,
    "BlobGasUsed": 1,
    "Withdrawals": 1,
    "WithdrawalsRoot": 1,
    "Raw": 1,
    "RawHeader": 1,
    "TotalDifficulty": 1,
    "LogsBloom": 1,
    "ExtraData": 1,
    "Ommers": 1,
    "OmmerCount": 1,
    "OmmerHash": 1,
    "ReceiptsRoot": 1,
    "StateRoot": 1,
    "TransactionsRoot": 1,
    "MixHash": 1,
    "Timestamp": 1,
    "Difficulty": 1,
    "Parent": 1,
    "NextBaseFeePerGas": 1,
    "BaseFeePerGas": 1,
    "GasUsed": 1,
    "GasLimit": 1,
    "Number": 1,
    "YParity": 1,
    "AccessList": 1,
    "Type": 1,
    "BlobGasPrice": 1,
    "CumulativeGasUsed": 1,
    "Index": 1,
    "Value": 1,
    "EffectiveTip": 1,
    "BlobVersionedHashes": 1,
    // 
    "MaxFeePerBlobGas": 1,
    "MaxFeePerGas": 1,
    "EffectiveGasPrice": 1,
    "Gas": 1,
    "InputData": 1
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Hash,6364,1,3182,3182.0,False
1,Number,1446,1,723,723.0,False
2,Value,1096,1,548,548.0,False
3,Type,1049,1,525,525.0,False
4,Block,770,1,385,385.0,False
5,Nonce,712,1,356,356.0,False
6,Account,442,1,221,221.0,False
7,Transaction,852,2,426,213.0,False
8,Gas,420,1,210,210.0,False
9,Difficulty,387,1,194,194.0,False


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,151.793103,1.051724,147.396552


44.15217391304348
0.002173913043478261 0.005150264264495968 45.81266154855137 42.521449789745766


In [128]:
folder = 'e:\\phd\\test_repos_graphql2\\exo\\'
words = {
    "Components": 2,
    "FindManifest": 1,
    "WatchJob": 1,
    "StackByID": 1,
    "DestroyComponents": 1,
    "UpdateComponent": 2,
    "DestroyStack": 1,
    "FinishTask": 1,
    "AcquireTask": 1,
    "DisposeResource": 1,
    "UpdateResource": 2,
    //
    "RestartWorkspaceComponents": 1,
    "CreateWorkspace": 2,
    "CreateProject": 1,
    "ResourceByID": 1,
    "MakeManifest": 1,
    "FindEntity": 1,
    "Stack": 1,
    "Manifest": 1,
    "BuildWorkspace": 1,
    "DestroyWorkspace": 1,
    "TaskByID": 1,
    "DisplayName": 1,
    "ForgetResource": 1,
    "ReconcileComponent": 1,
    "InitializeResource": 1,
    "RefreshResource": 1,
    "RefreshCluster": 1,
    "SetWorkspaceStack": 1,
    "Build": 1,
    "Project": 1,
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Stack,85,1,43,43.0,False
1,DisplayName,75,1,38,38.0,False
2,Build,54,1,27,27.0,False
3,Components,75,2,38,19.0,False
4,Project,36,1,18,18.0,True
5,Manifest,26,1,13,13.0,True
6,CreateProject,5,1,3,3.0,True
7,CreateWorkspace,7,2,4,2.0,True
8,DestroyComponents,3,1,2,2.0,True
9,UpdateResource,6,2,3,1.5,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,6.806452,1.129032,6.048387


10.148936170212766
0.0008530183727034121 0.002098704197408395 10.816052337778435 9.494276860894146


In [129]:
folder = 'e:\\phd\\test_repos_graphql2\\HydroAPI\\'
words = {
    "Merit": 1,
    "Strike": 1,
    "Giveaway": 1,
    "Server": 1,
    "User": 1,
    "Msgs": 1,
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Server,109,1,55,55.0,False
1,User,95,1,48,48.0,False
2,Giveaway,54,1,27,27.0,False
3,Merit,44,1,22,22.0,False
4,Strike,44,1,22,22.0,False
5,Msgs,7,1,4,4.0,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,29.666667,1.0,29.666667


41.0
0.0015384615384615385 0.00363905325443787 42.173964497041425 39.847041420118344


In [130]:
folder = 'e:\\phd\\test_repos_graphql2\\mattermost-plugin-playbooks\\'
words = {
    "UpdateRunTaskActions": 1,
    "ChangeRunOwner": 1,
    "RemoveRunParticipants": 1,
    "AddRunParticipants": 1,
    "UpdateRun": 1,
    "SetRunFavorite": 1,
    "RemovePlaybookMember": 2,
    "AddPlaybookMember": 2,
    "DeleteMetric": 2,
    "UpdateMetric": 2,
    "AddMetric": 2,
    "UpdatePlaybook": 2,
    "UpdatePlaybookFavorite": 1,
    "Runs": 1,
    "Run": 1,
    "Playbooks": 1,
    "Playbook": 1
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Run,463,1,232,232.0,False
1,Playbook,353,1,177,177.0,False
2,Playbooks,180,1,90,90.0,False
3,Runs,55,1,28,28.0,False
4,UpdateRunTaskActions,7,1,4,4.0,True
5,UpdatePlaybook,12,2,6,3.0,True
6,SetRunFavorite,3,1,2,2.0,True
7,ChangeRunOwner,4,1,2,2.0,True
8,UpdateMetric,7,2,4,2.0,True
9,AddMetric,7,2,4,2.0,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,33.352941,1.352941,32.647059


7.7924528301886795
0.0008733624454148472 0.0022445224156671306 8.49463242249241 7.10398483758747


In [131]:
folder = 'e:\\phd\\test_repos_graphql2\\SlabAPI\\'
words = {
    "GetViewCount": 2,
    "GetView": 1,
    "GetAllViews": 1,
    "GetUser": 1,
    "GetTrigger": 1,
    "GetAllTriggers": 1,
    "GetAllTickets": 1,
    "GetOrganization": 1,
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,GetTrigger,11,1,6,6.0,True
1,GetOrganization,11,1,6,6.0,True
2,GetView,10,1,5,5.0,True
3,GetUser,10,1,5,5.0,True
4,GetAllTriggers,7,1,4,4.0,True
5,GetAllTickets,7,1,4,4.0,True
6,GetAllViews,6,1,3,3.0,True
7,GetViewCount,10,2,5,2.5,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,4.75,1.125,4.4375


26.454545454545453
0.0016666666666666668 0.0040304709141274235 27.743972970704274 25.18875598086124


In [132]:
folder = 'e:\\phd\\test_repos_graphql2\\sturdy\\'
words = {
    "Integrations": 1,
    "UpdatedGitHubPullRequestStatuses": 1,
    "PullCodebase": 1,
    "CreateOrUpdateCodebaseRemote": 1,
    "UpdateInstallation": 1,
    "DeleteIntegration": 1,
    "SetupGitHubRepository": 1,
    "Activity": 1,
    "DownloadZip": 1,
    "Changes": 1,
    "Members": 2,
    "LastUsedView": 2,
    "UpdatedOrganization": 1,
    "UpdatedSuggestion": 1,
    "UpdatedView": 1,
    "UpdatedWorkspaceDiffs": 1,
    "GenerateKeyPair": 1,
    "RemovePatches": 2,
    "DismissSuggestionHunks": 1,
    "AddPublicKey": 1,
    "WatchWorkspace": 1,
    "RequestReview": 1,
    "UpdateACL": 1,
    "ArchiveNotifications": 1,
    "AddUserToCodebase": 1,
    "VerifyEmail": 2,
    "UpdateUser": 2,
    "DeleteComment": 1,
    "LandWorkspaceChange": 2,
    "Comment": 3,
    "Workspaces": 2,
    "ResolvedBy": 1,
    "Notifications": 1,
    "UpdateComment": 1,
    "CreateComment": 1,
    "UpdateCodebase": 1,
    "ReportWorkspacePresence": 1,
    "DismissSuggestion": 1,
    "UpdateOrganization": 1,
    "UpdatedWorkspacesStatuses": 1,
    "CreateOrUpdateGitHubPullRequest": 1,
    "TriggerInstantIntegration": 1,
    "PushCodebase": 1,
    "UpdatedGitHubPullRequest": 1,
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Activity,43,1,22,22.000000,False
1,Comment,94,3,47,15.666667,False
2,LandWorkspaceChange,22,2,11,5.500000,True
3,Members,19,2,10,5.000000,True
4,AddPublicKey,7,1,4,4.000000,True
5,ResolvedBy,6,1,3,3.000000,True
6,Changes,6,1,3,3.000000,True
7,VerifyEmail,7,2,4,2.000000,True
8,UpdatedGitHubPullRequestStatuses,3,1,2,2.000000,True
9,CreateComment,4,1,2,2.000000,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,3.5,1.204545,2.390152


7.353383458646617
0.000689935064935065 0.001727789256198347 7.8990224787174546 6.818122980488411


In [133]:
folder = 'e:\\phd\\test_repos_graphql2\\toolkit-for-cardano\\'
words = {
    "WalletDelegate": 1,
    "WalletRegister": 1,
    "WalletFund": 1,
    "WalletCreate": 1,
    "SendFunds": 1,
    "TxSubmit": 1,
    "TxSign": 1,
    "TxBuild": 1,
    "Mint": 1,
    "Wallets": 1,
    "Utxos": 2,
    "TxFee": 1,
}
analyze(folder, words)

,Word,Count Search,Count Markup,Count Search 2,Ratio,Count Search/2<20
0,Utxos,22,2,11,5.5,True
1,Mint,9,1,5,5.0,True
2,WalletDelegate,2,1,1,1.0,True
3,WalletRegister,2,1,1,1.0,True
4,WalletFund,1,1,1,1.0,True
5,WalletCreate,1,1,1,1.0,True
6,SendFunds,1,1,1,1.0,True
7,TxSubmit,2,1,1,1.0,True
8,TxSign,2,1,1,1.0,True
9,TxBuild,1,1,1,1.0,True


,Word,Count Search 2,Count Markup,Ratio
0,MEAN,2.166667,1.083333,1.708333


9.214285714285714
0.0007831325301204819 0.0019248076643925097 9.826355680899155 8.613632499014992


In [135]:
data = global_array
# Определяем границы интервалов
bins = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 50, 100, 7000]

# Задаем названия интервалов
intervals = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
    '11-15', '16-20', '21-30', '31-50', '51-100', '>100'
]

# Считаем количество элементов в каждом интервале
counts, _ = np.histogram(data, bins=bins)

# Создаем словарь {интервал: количество}
result = dict(zip(intervals, counts))
print(result)

{'1': 56, '2': 50, '3': 22, '4': 20, '5': 7, '6': 7, '7': 13, '8': 13, '9': 11, '10': 24, '11-15': 8, '16-20': 12, '21-30': 17, '31-50': 22, '51-100': 43, '>100': 0}


In [150]:
print(len(data[data==1]))
print(len(data[(data>1)&(data<=4)]))
print(len(data[(data>5)&(data<=10)]))
print(len(data[(data>10)]))

56
91
51
119


In [109]:
# folder = 'e:\\phd\\test_repos\\modules\\'
# words = {
#     "CompleteFollowup": 1, 
#     "ReAssignFollowup": 2,
#     "RemovePharmacy": 1,
#     "SetABVariant": 1,
#     "BlockPatient": 1,
#     "SetMediaRead": 1,
#     "ValidateAppointmentInsurance": 3,
#     "UpdateProviderInsurance": 1,
#     "EditStateCompanies": 1,
#     "UpdateEncounterScreenedStatus": 1,
#     "StartPaymentTransaction": 1,
#     "SubscribeToPhoneCallEvents": 1,
#     "SetAppointmentType": 1,
#     "MakeCardDefault": 2,
#     "DeletePatientNote": 1,
#     "UpdatePatientNote": 1,
#     "SetVisitTypeOrders": 2,
#     "SetVisitTypeVisibility": 2,
#     "AddICD10MedicalCode": 2,
#     "AddMedicalCodes": 1,
#     "InviteClient": 1,
#     "DeleteEvent": 1,
#     "NeedFollowup": 1,
#     "TwilioInfo": 1,
#     "CurrentTime": 1,
#     "Person": 1,
#     "PHIData": 1,
#     "Clients": 1,
#     "Messages": 1,
#     "PastVisits": 1,
#     "Appointment": 1,
#     "CheckInData": 1,
#     "NoteTemplate": 1,
#     "VisitType": 1,
#     "ScheduleVisit": 1,
#     "UpdateAvailability": 1,
#     "AssignQuestionnaire": 1,
#     "PublishForm": 1,
#     "LoadFollowupByPatients": 1,
#     "InsuranceCosts": 1,
#     "PatientDetails": 1,
#     "ConsentForms": 1
# }
# analyze(folder, words)

In [95]:
# folder = 'e:\\phd\\test_repos_graphql2\\bramble\\'
# words = {
#     "UploadGadgetFile": 1, 
#     "UploadGizmoFile": 1, 
#     "Gizmo": 2, 
#     "Gizmos": 2, 
#     "Delay": 1, 
#     "Gizmos": 2, 
#     "RandomGizmo": 1,
# }
# analyze(folder, words)

In [110]:
# folder = 'e:\\phd\\test_repos_graphql2\\go-graphql-starter\\'
# words = {
#     "Signup": 1,
#     "User": 1,
#     "Login": 1,
#     "Viewer": 1,
# }
# analyze(folder, words)

In [105]:
# folder = 'e:\\phd\\test_repos\\backend\\fsm-gateway\\'
# words = {
#     "GetSession": 1, 
#     "GetDocumentData": 1,
#     "GetDocumentText": 1,
#     "GetConsentList": 1,
#     "StartSession": 1,
#     "SendDocumentData": 1,
#     "FinishSession": 1,
#     "StartAuthSessionByPhone": 1,
#     "GetJwtByPhoneCode": 1,
#     "GetJwtByRefreshToken": 1
# }
# df = analyze(folder, words)

In [106]:
# folder = 'e:\\phd\\test_repos\\LP\\delivery-offering\\'
# words = {
#     "DeliveryTimeslots": 2, 
#     "DeliveryTimeslot": 2, 
#     "DeliveryTypes": 2, 
#     "ReserveDeliveryTimeslot": 2,
#     "LookupDeliveryOffers": 2,
#     "DeliveryOfferByID": 2,
#     "AcceptDeliveryOffer": 2,
#     "ValidateDeliveryOffer": 2,
# }
# analyze(folder, words)

In [107]:
# folder = 'e:\\phd\\test_repos\\LP\\warehouses\\'
# words = {
#     "TerminateWarehouses": 2, 
#     "UpdateWarehouses": 2, 
#     "CreateWarehouses": 2, 
#     "Warehouses": 2,
# }
# analyze(folder, words)

In [108]:
# folder = 'e:\\phd\\test_repos\\LP\\channel-profile\\'
# words = {
#     "CreateChannelProfile": 2, 
#     "UpdateChannelProfile": 2, 
#     "DeleteChannelProfile": 2, 
#     "ChannelProfiles": 2,
#     "ChannelsByFilter": 2,
# }
# analyze(folder, words)